# 06 · Data audit: Symcat and MedDialog-EN

**SimClinic, CS425 Group 7 (data).** This notebook re-runs the audit behind Table 1 of the proposal from the raw files, so every number in it can be checked:

1. **Symcat** (via symcat-to-synthea): size, the 12-symptom cap, tied percentages, and the quality filter that decides which conditions can become case cards.
2. **MedDialog-EN** (`data/raw/diagnose-me/en_medical_dialog.json`, the full English scrape): dialogue structure, how rarely doctors ask questions, stock greetings, and the length of patient messages.

It writes nothing. Notebook 07 builds the case cards, lexicon checks and generated dialogues from these decisions.

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW_SYMCAT = ROOT / "data" / "raw" / "symcat"
RAW_MEDDIALOG = ROOT / "data" / "raw" / "diagnose-me" / "en_medical_dialog.json"
SAMPLE = {"healthcaremagic": 2000, "icliniq": 500}     # the proposal's sample sizes
SEED = 42

## 1. Symcat
The CSVs are downloaded from symcat-to-synthea if they are missing. `simclinic/symcat.py` parses them by anchoring on the Symcat URLs, because the column a field lands in shifts between rows.

In [2]:
from collections import Counter
import pandas as pd
from simclinic import symcat

pd.set_option("display.max_colwidth", 120)
symcat.download(RAW_SYMCAT)
conditions = symcat.parse_conditions(RAW_SYMCAT / "symcat-801-diseases.csv")

def max_tied(c):
    return max(Counter(c.symptoms.values()).values()) if c.symptoms else 0

passing = [n for n, c in conditions.items() if symcat.passes_quality_filter(c)]
audit = pd.DataFrame([
    ("conditions", len(conditions), 801),
    ("symptoms", symcat.count_symptoms(RAW_SYMCAT / "symcat-474-symptoms.csv"), 474),
    ("condition-symptom links", sum(len(c.symptoms) for c in conditions.values()), 9162),
    ("conditions with exactly 12 symptoms (the cap)", sum(len(c.symptoms) == 12 for c in conditions.values()), 687),
    ("conditions with >= 5 symptoms at the same percentage", sum(max_tied(c) >= 5 for c in conditions.values()), 296),
    ("conditions passing the quality filter", len(passing), 391),
], columns=["", "here", "proposal"]).set_index("")
assert (audit["here"] == audit["proposal"]).all()
audit

,here,proposal
,,
conditions,801,801
symptoms,474,474
condition-symptom links,9162,9162
conditions with exactly 12 symptoms (the cap),687,687
conditions with >= 5 symptoms at the same percentage,296,296
conditions passing the quality filter,391,391


Ties are the warning sign. Symcat's percentages come from patient records, so many symptoms sharing exactly one percentage means a condition was described by a handful of records. Flu is well described; dengue lists fever and sore throat at 100% and then ten symptoms at exactly 32%, including "pain during pregnancy" and "excessive anger". Typhoid has nearly the same list.

In [3]:
def show(names):
    return pd.DataFrame({n: pd.Series([f"{s} {round(p * 100)}%" for s, p in conditions[n].symptoms.items()])
                         for n in names}).fillna("")

dengue, typhoid = set(conditions["Dengue fever"].symptoms), set(conditions["Typhoid fever"].symptoms)
print(f"dengue and typhoid share {len(dengue & typhoid)} of their {len(dengue)} symptoms")
show(["Flu", "Dengue fever", "Typhoid fever"])

dengue and typhoid share 11 of their 12 symptoms


,Flu,Dengue fever,Typhoid fever
0,Fever 91%,Sore throat 100%,Fever 100%
1,Cough 87%,Fever 100%,Shoulder cramps or spasms 32%
2,Sore throat 63%,Shoulder cramps or spasms 32%,Excessive anger 32%
3,Nasal congestion 63%,Ankle pain 32%,Ankle pain 32%
4,Ache all over 56%,Wrist pain 32%,Wrist pain 32%
5,Flu-like syndrome 52%,Pain during pregnancy 32%,Pain during pregnancy 32%
6,Headache 52%,Joint stiffness or tightness 32%,Facial pain 32%
7,Vomiting 47%,Excessive anger 32%,Joint stiffness or tightness 32%
8,Chills 38%,Pain or soreness of breast 32%,Pain or soreness of breast 32%
9,Coryza 37%,Knee lump or mass 32%,Knee lump or mass 32%


**Quality filter** (`symcat.passes_quality_filter`): keep a condition only if no five of its symptoms share a percentage and at least three are reported by 30% or more of patients. The histogram shows what is rejected and why.

In [4]:
reasons = Counter()
for c in conditions.values():
    if not c.symptoms:
        reasons["no symptoms parsed"] += 1
    elif max_tied(c) >= 5:
        reasons[">= 5 tied percentages"] += 1
    elif sum(p >= 0.30 for p in c.symptoms.values()) < 3:
        reasons["< 3 symptoms at >= 30%"] += 1
    else:
        reasons["passes"] += 1
pd.Series(reasons).to_frame("conditions")

,conditions
< 3 symptoms at >= 30%,114
passes,391
>= 5 tied percentages,296


**Demographics.** Symcat gives age and sex as odds ratios per condition (used to weight the persona's age group and sex), and race as US census groups. We use Singapore's resident ethnic groups instead (Census 2020: Chinese 74%, Malay 13.5%, Indian 9%, Others 3.5%).

In [5]:
print(f"{sum(bool(c.age_odds) for c in conditions.values())} conditions with age odds, "
      f"{sum(bool(c.sex_odds) for c in conditions.values())} with sex odds")
pd.DataFrame({"age odds": conditions["Flu"].age_odds, "": None}).drop(columns="").T

801 conditions with age odds, 801 with sex odds


,< 1 years,1-4 years,5-14 years,15-29 years,30-44 years,45-59 years,60-74 years,75+ years
age odds,1.3,2.2,2.9,1.2,0.9,0.5,0.3,0.3


## 2. MedDialog-EN
The proposal counted 257,332 dialogues and 514,664 utterances in the Hugging Face data viewer, exactly two utterances per dialogue. The local scrape has one row per dialogue (`Description`, `Patient`, `Doctor`), so it has the same structure. Its row count is slightly higher: it contains duplicates, which notebook 02 removes.

In [6]:
import re

df = pd.read_json(RAW_MEDDIALOG)
for col in ("Description", "Doctor", "Patient"):
    df[col] = df[col].fillna("").astype(str)
# iCliniq questions are titled "Q. ..."; the rest is HealthCareMagic.
df["source"] = df["Description"].str.startswith("Q.").map({True: "icliniq", False: "healthcaremagic"})
print(f"{len(df):,} dialogues = {len(df):,} patient messages + {len(df):,} doctor replies "
      f"(proposal, HF viewer: 257,332 dialogues, 514,664 utterances)")
df["source"].value_counts().to_frame("dialogues")

257,469 dialogues = 257,469 patient messages + 257,469 doctor replies (proposal, HF viewer: 257,332 dialogues, 514,664 utterances)


,dialogues
source,
healthcaremagic,226405
icliniq,31064


Each item is one online question and one answer, not an interview. To see whether the doctor side could teach history taking, we sample the proposal's 2,000 HealthCareMagic and 500 iCliniq threads and measure:
- how many doctor replies **ask anything** (contain a question mark);
- how many **open with a stock greeting** ("Hi", "Hello", "Thanks for your query", "Welcome to ...");
- how long the **patient messages** are.

The proposal measured the ChatDoctor copies of the two sites; this is the original scrape, so the shares differ by a few points but lead to the same decision.

In [7]:
GREETING = re.compile(r"^\s*(?:hi|hello|hey|dear|thanks?|thank you|welcome|greetings|respected|good (?:morning|day|evening))\b", re.I)
sample = pd.concat([df[df["source"] == s].sample(n, random_state=SEED) for s, n in SAMPLE.items()])
proposal = {"healthcaremagic": (5.4, 83, 69), "icliniq": (10.6, 99, 69)}
rows = []
for src, g in sample.groupby("source", sort=False):
    rows.append({"source": src, "sampled": len(g),
                 "doctor asks a question (%)": round(100 * g["Doctor"].str.contains(r"\?").mean(), 1),
                 "doctor opens with a greeting (%)": round(100 * g["Doctor"].str.match(GREETING).mean(), 1),
                 "patient words (median)": g["Patient"].str.split().str.len().median(),
                 "proposal (question % / greeting % / words)": "%s / %s / %s" % proposal[src]})
pd.DataFrame(rows).set_index("source")

,sampled,doctor asks a question (%),doctor opens with a greeting (%),patient words (median),proposal (question % / greeting % / words)
source,,,,,
healthcaremagic,2000,5.0,77.4,69.0,5.4 / 83 / 69
icliniq,500,14.4,100.0,69.0,10.6 / 99 / 69


In [8]:
# What the doctor side looks like, and why the patient side is the useful one.
for _, r in sample.sample(3, random_state=1).iterrows():
    print("PATIENT:", r["Patient"][:300]); print("DOCTOR :", r["Doctor"][:300]); print("-" * 100)

In [9]:
# Patient messages are informal, with hedging and typos: the style we want for the patient role.
words = df["Patient"].str.split().str.len()
hedges = df["Patient"].str.contains(r"\b(?:i think|maybe|not sure|kind of|sort of|i guess|probably|a bit)\b", case=False)
pd.Series({"patient words p5 / median / p95": f"{words.quantile(.05):.0f} / {words.median():.0f} / {words.quantile(.95):.0f}",
           "patient messages with a hedge (%)": round(100 * hedges.mean(), 1)}).to_frame("value")

,value
patient words p5 / median / p95,42 / 69 / 171
patient messages with a hedge (%),12.0


## 3. Decisions
| Source | Finding | Decision |
|---|---|---|
| Symcat | 801 conditions, 474 symptoms, 9,162 links; lists capped at 12; 296 conditions with ≥ 5 tied percentages | Case cards come from the 391 conditions that pass the quality filter. We hand-pick 59 common Singapore GP conditions from them and add team-written dengue and HFMD cards (`simclinic/team_conditions.json`), because Symcat's dengue entry fails the filter and Symcat has no HFMD. |
| Symcat | Clinical symptom names ("coryza") | Lay-phrasing lexicon (`simclinic/symptom_lexicon.json`). |
| Symcat | US census race groups | Singapore ethnic groups in the persona. |
| MedDialog-EN | One patient message and one doctor reply per dialogue; doctors rarely ask questions and mostly open with a greeting | The doctor's questioning comes from a SOCRATES-style history-taking checklist in the generation prompt, not from MedDialog. |
| MedDialog-EN | Informal patient messages (median 69 words) | Cleaned patient messages (notebook 02, train split only) are style examples in the generation prompt. The data stays on our machines, as the source sites do not allow republishing. |

No source has multi-turn consultations tied to a known disease, so notebook 07 generates them from the case cards.